# Tax Q&A Evaluation Pipeline

This notebook demonstrates using GoTaglio to evaluate LLM responses to US indirect tax questions.

## 1. Setup

Load the pipeline and test cases.

> **Note:** Test cases in `data/cases.yaml` were created using product categories from the UNSPSC (United Nations Standard Products and Services Code) taxonomy. The full UNSPSC dataset is available in `data/unspsc_codes.csv` for reference when adding new test cases.

In [ ]:
import yaml
from tax import tax_pipeline_spec  # type: ignore
from gotaglio.gotag import Gotaglio

# Load cases from YAML
with open("data/cases.yaml") as f:
    all_cases = yaml.safe_load(f)

print(f"Loaded {len(all_cases)} cases")
print(f"Complexity levels: {set(c.get('complexity', 'unknown') for c in all_cases)}")

# Initialize GoTaglio
gt = Gotaglio([tax_pipeline_spec])

## 2. Run with Mock Model

Test the pipeline with `perfect` model (returns expected values - no LLM needed).

In [ ]:
# Run a sample of 3 cases with mock model
sample_cases = all_cases[:3]

result = gt.run(
    "tax",
    sample_cases,
    {
        "prepare.template": "data/template.txt",
        "infer.model.name": "perfect",
    },
    save=True,
)
gt.format(result)

## 3. Run with Real LLM

Configure your model in `/workspaces/gotaglio/models.json` and `.credentials.json`, then run:

In [ ]:
# Run all cases with real LLM
# Change model name to match your models.json configuration

result = gt.run(
    "tax",
    all_cases,
    {
        "prepare.template": "data/template.txt",
        "infer.model.name": "niksac-gpt-4.1",  # Change to your model
    },
    save=True,
)
gt.format(result)

## 4. View Run History & Compare

Each run is saved to `logs/` with a UUID for versioning and comparison.

In [ ]:
# View run history
gt.history()

In [ ]:
# Access run metadata
print(f"Run UUID: {result['uuid']}")
print(f"Model: {result['metadata']['pipeline']['config']['infer']['model']['name']}")
print(f"Elapsed: {result['metadata']['elapsed']}")

In [ ]:
# View detailed results for first case
import json

case_result = result['results'][0]
print(f"Case: {case_result['case']['uuid']}")
print(f"Question: {case_result['case']['turns'][0]['user']}")
print(f"Status: {'PASSED' if case_result.get('passed') else 'FAILED/ERROR'}")

if 'turns' in case_result and case_result['turns']:
    turn = case_result['turns'][0]
    if 'stages' in turn and 'extract' in turn['stages']:
        print(f"\nLLM Response:")
        print(json.dumps(turn['stages']['extract'], indent=2))

## 5. Filter Cases by Complexity

Run subsets of cases by filtering on keywords or complexity.

In [ ]:
# Filter by complexity
basic_cases = [c for c in all_cases if c.get('complexity') == 'basic']
advanced_cases = [c for c in all_cases if c.get('complexity') in ['advanced', 'strategic']]

print(f"Basic cases: {len(basic_cases)}")
print(f"Advanced cases: {len(advanced_cases)}")

# Filter by keyword
texas_cases = [c for c in all_cases if 'texas' in c.get('keywords', [])]
software_cases = [c for c in all_cases if 'software' in c.get('keywords', [])]

print(f"Texas cases: {len(texas_cases)}")
print(f"Software cases: {len(software_cases)}")